# Instamart Customer Intelligence & Support Copilot
## Notebook 1 of 2 — Part A (Data Foundation) and Part B (Customer Segments & Lapse Prediction)

**Scenario:** synthetic quick-commerce data (customers, products, orders) for six Indian cities. All data is fictional.

| Part | What this notebook does | Main outputs |
|---|---|---|
| **A** | Audit the raw exports, clean them, store them in SQLite, validate with checkpoints, query with SQL and cross-check with pandas | `data/clean/*.csv`, `data/instamart.db`, `sql/queries.sql` |
| **B** | RFM features, K-means segments, PCA, lapse-prediction model (leakage-aware), retention list | `outputs/customer_segments.csv`, `outputs/lapse_model.joblib`, `outputs/retention_list.csv` |

**Run order:** run every cell from top to bottom (*Run All*). Notebook 2 (Parts C and D) needs the files created here, so run this notebook first.

## 0. Project setup
Locate the repository root and make it the working directory so that relative paths such as `data/raw/` work no matter where the notebook file is saved.

In [2]:

import os
import sys
import subprocess
from pathlib import Path

# Check current and parent directories
CURRENT = Path.cwd()

if (CURRENT / "generate.py").exists():
    ROOT = CURRENT
elif (CURRENT.parent / "generate.py").exists():
    ROOT = CURRENT.parent
else:
    raise FileNotFoundError(
        f"generate.py not found in {CURRENT} or {CURRENT.parent}"
    )

# Set project root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# Verification
print("Project root:", ROOT)
print("generate.py exists:", (ROOT / "generate.py").exists())
print("Project files:", [p.name for p in ROOT.iterdir()])




Project root: c:\Users\Lavanya Ramakrishnan\OneDrive\Desktop\Capstone project
generate.py exists: True
Project files: ['.gitignore', '.venv', 'data', 'generate.py', 'mock_llm.py', 'notebooks', 'outputs', 'requirements.txt', 'run_project.py', 'sql', 'src', 'state', '__pycache__']


# PART A — Data Foundation: Clean, Store, Query

## Task 1. Generate the raw data
The supplied `generate_data.py` is deterministic (fixed random seed). It writes the raw CSV files into `data/raw/` and the policy text files into `data/policies/`. Its last line of output must be exactly:

`generated: customers=1200 products=40 orders_raw=13084 tickets=164`

In [3]:
result = subprocess.run([sys.executable, "generate.py"], capture_output=True, text=True, check=True)
print(result.stdout.strip())
assert result.stdout.strip().splitlines()[-1] == "generated: customers=1200 products=40 orders_raw=13084 tickets=164"

raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

generated: customers=1200 products=40 orders_raw=13084 tickets=164
Raw files: ['agent_scenarios.csv', 'customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


## Task 2. Audit the raw data
Before changing anything, look at the raw exports: shape, data types, blanks, duplicates and impossible values. Every cleaning rule in Task 3 and Task 4 comes from something found here.

In [4]:
import pandas as pd
import os
# TASK 2: AUDIT RAW DATA
# Load raw datasets
data_path = "data/raw/"
customers_data = pd.read_csv(data_path + "customers.csv")
products = pd.read_csv(data_path + "products.csv")
orders = pd.read_csv(data_path + "orders.csv")

datasets = {
    "Customers": customers_data,
    "Products": products,
    "Orders": orders
}

In [5]:
for name, df in datasets.items():
    print("\n==========", name, "==========")
    print("Shape:", df.shape)
    print("\nData Types:")
    print(df.dtypes)
    missing_count = df.isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100
    missing_report = pd.DataFrame({
        "Missing Count": missing_count,
        "Missing Percentage": missing_percentage
    })
    print("\nMissing Values:")
    print(missing_report)


========== Customers ==========
Shape: (1200, 5)

Data Types:
customer_id        str
city               str
signup_date        str
age            float64
gender             str
dtype: object

Missing Values:
             Missing Count  Missing Percentage
customer_id              0            0.000000
city                     0            0.000000
signup_date              0            0.000000
age                     61            5.083333
gender                  30            2.500000

========== Products ==========
Shape: (40, 4)

Data Types:
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object

Missing Values:
              Missing Count  Missing Percentage
product_id                0                 0.0
product_name              0                 0.0
category                  0                 0.0
price_inr                 0                 0.0

========== Orders ==========
Shape: (13084, 9)

Data Types:
order_id                str
c

In [6]:
print("\nExact duplicate rows:", orders.duplicated().sum())
print("Pending orders:", (orders["status"] == "pending").sum())
print("Distinct order IDs:", orders["order_id"].nunique())
numeric_amounts = pd.to_numeric(orders["order_amount"], errors="coerce")
print("Non-numeric or blank order amounts:", numeric_amounts.isna().sum())
print("Negative delivery minutes:", (orders["delivery_minutes"] < 0).sum())
print("Blank delivery minutes:", orders["delivery_minutes"].isna().sum())
print("\nBlank ages:", customers_data["age"].isna().sum())
invalid_ages = (customers_data["age"] < 10) | (customers_data["age"] > 100)
print("Ages outside 10-100:", invalid_ages.sum())
print("Distinct raw city spellings:", customers_data["city"].nunique())
print("Distinct non-blank raw gender spellings:", customers_data["gender"].dropna().nunique())
print("Blank genders:", customers_data["gender"].isna().sum())


Exact duplicate rows: 119
Pending orders: 656
Distinct order IDs: 12309
Non-numeric or blank order amounts: 120
Negative delivery minutes: 106
Blank delivery minutes: 2541

Blank ages: 61
Ages outside 10-100: 6
Distinct raw city spellings: 24
Distinct non-blank raw gender spellings: 10
Blank genders: 30


## Task 3. Clean the customers table
Rules: standardise `city` (strip + title case) → standardise `gender` (`male` / `female` / `unknown`) → mask impossible ages (outside 10–100) → fill missing ages with the **city median**. City is standardised *first*, otherwise the city-wise median would be calculated over split groups such as `chennai` and `CHENNAI`.

In [7]:
print(customers_data.head(10))
customers_data.info()
print("Duplicate rows:", customers_data.duplicated().sum())
print("Duplicate customer IDs:", customers_data["customer_id"].duplicated().sum())
print(customers_data["age"].describe())
invalid_age = customers_data[
    (customers_data["age"] < 10) | (customers_data["age"] > 100)
]
print("Invalid age count:", len(invalid_age))
print(invalid_age[["customer_id", "age"]])

  customer_id       city signup_date   age  gender
0       C0001    Chennai  2024-01-13  35.0    MALE
1       C0002      DELHI  2024-02-17  31.0    MALE
2       C0003  bengaluru  2025-02-18  18.0       F
3       C0004  Bengaluru  2024-06-21  24.0  FEMALE
4       C0005  Bengaluru  2024-10-01  25.0  FEMALE
5       C0006   Chennai   2024-01-24  60.0    Male
6       C0007  Hyderabad  2025-03-03  41.0  female
7       C0008    CHENNAI  2024-05-05  28.0       M
8       C0009       Pune  2024-06-10  43.0  Female
9       C0010      Delhi  2024-11-25  47.0  FEMALE
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  1200 non-null   str    
 1   city         1200 non-null   str    
 2   signup_date  1200 non-null   str    
 3   age          1139 non-null   float64
 4   gender       1170 non-null   str    
dtypes: float64(1), str(4)
memory usage: 76.6 KB
D

In [8]:
# Standardize city before city-wise age imputation
customers_cleaned = customers_data.copy()
customers_cleaned["city"] = customers_cleaned["city"].str.strip().str.title()
city_changes = (customers_data["city"] != customers_cleaned["city"]).sum()
print("\nCity values standardized:", city_changes)
print("\nCities after standardization:")
print(customers_cleaned["city"].value_counts())
print("\nNumber of unique cities:", customers_cleaned["city"].nunique())


City values standardized: 476

Cities after standardization:
city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
Name: count, dtype: int64

Number of unique cities: 6


In [9]:
# Standardize gender
gender_mapping = {
    "female": "female",
    "f": "female",
    "male": "male",
    "m": "male"
}
customers_cleaned["gender"] = (
    customers_cleaned["gender"]
    .str.strip()
    .str.lower()
    .map(gender_mapping)
    .fillna("unknown")
)
blank_gender = customers_data["gender"].isna().sum()
gender_changes = (
    customers_data["gender"].notna()
    & (customers_data["gender"] != customers_cleaned["gender"])
).sum()
print("Gender values standardized (non-blank):", gender_changes)
print("Blank genders mapped to unknown:", blank_gender)
print("\nGender counts:")
print(customers_cleaned["gender"].value_counts())

Gender values standardized (non-blank): 920
Blank genders mapped to unknown: 30

Gender counts:
gender
female     610
male       560
unknown     30
Name: count, dtype: int64


In [10]:
# Mask invalid ages and fill missing ages using city median
original_blank_ages = customers_cleaned["age"].isna().sum()
invalid_age_count = (
    (customers_cleaned["age"] < 10) | (customers_cleaned["age"] > 100)
).sum()
print("\nOriginal blank ages:", original_blank_ages)
print("Age outliers masked:", invalid_age_count)
customers_cleaned.loc[
    (customers_cleaned["age"] < 10) | (customers_cleaned["age"] > 100),
    "age"
] = pd.NA
missing_before_imputation = customers_cleaned["age"].isna().sum()
print("Missing ages before median imputation:", missing_before_imputation)
customers_cleaned["age"] = (
    customers_cleaned["age"].fillna(
        customers_cleaned.groupby("city")["age"].transform("median")
    )
)
missing_after_imputation = customers_cleaned["age"].isna().sum()
print("Ages filled using city median:", missing_before_imputation - missing_after_imputation)
print("Missing ages after median imputation:", missing_after_imputation)

customers_cleaned["signup_date"] = pd.to_datetime(customers_cleaned["signup_date"])
print("\nFinal Age Summary:")
print(customers_cleaned["age"].describe())
print("\nMinimum age:", customers_cleaned["age"].min())
print("Maximum age:", customers_cleaned["age"].max())


Original blank ages: 61
Age outliers masked: 6
Missing ages before median imputation: 67
Ages filled using city median: 67
Missing ages after median imputation: 0

Final Age Summary:
count    1200.000000
mean       39.634583
std        12.046512
min        18.000000
25%        29.000000
50%        39.000000
75%        50.000000
max        60.000000
Name: age, dtype: float64

Minimum age: 18.0
Maximum age: 60.0


## Task 4. Clean the orders table
Rules, in order:
1. parse `order_ts` as a timestamp;
2. drop exact duplicate rows;
3. for repeated `order_id`s (an earlier *pending* record plus the final one) keep the **latest** timestamp;
4. convert `order_amount` to numeric (`N/A` / `error` become missing) and refill from product price × quantity × (1 − discount);
5. negative `delivery_minutes` become missing;
6. missing delivery time of **completed** orders is filled with the median of the product category;
7. cancelled orders must have no delivery time.

In [11]:
# TASK 4: CLEAN ORDERS
clean_orders_copy = orders.copy()

# Step 1: Convert timestamp
clean_orders_copy["order_ts"] = pd.to_datetime(clean_orders_copy["order_ts"])

# Step 2: Remove exact duplicates
rows_before = len(clean_orders_copy)
clean_orders_copy = clean_orders_copy.drop_duplicates()
print("\nExact duplicate order rows removed:", rows_before - len(clean_orders_copy))

# Step 3: Keep latest timestamp per order_id; stable sort preserves input order for ties
rows_before = len(clean_orders_copy)
clean_orders_copy = (
    clean_orders_copy
    .sort_values("order_ts", kind="stable")
    .drop_duplicates(subset="order_id", keep="last")
)
print("Older order_id records removed:", rows_before - len(clean_orders_copy))


Exact duplicate order rows removed: 119
Older order_id records removed: 656


In [12]:
# Step 4: Convert amounts and fill missing from product price
clean_orders_copy["order_amount"] = pd.to_numeric(
    clean_orders_copy["order_amount"], errors="coerce"
)
clean_orders_copy = clean_orders_copy.merge(
    products[["product_id", "category", "price_inr"]],
    on="product_id", how="left"
)
missing_amount = clean_orders_copy["order_amount"].isnull()
print("Missing order amounts to fill:", missing_amount.sum())
clean_orders_copy.loc[missing_amount, "order_amount"] = (
    clean_orders_copy.loc[missing_amount, "price_inr"]
    * clean_orders_copy.loc[missing_amount, "quantity"]
    * (1 - clean_orders_copy.loc[missing_amount, "discount_pct"] / 100)
).round(2)

# Step 5: Mask negative delivery minutes
negative_delivery_count = (clean_orders_copy["delivery_minutes"] < 0).sum()
print("Negative delivery minutes masked:", negative_delivery_count)
clean_orders_copy.loc[
    clean_orders_copy["delivery_minutes"] < 0, "delivery_minutes"
] = pd.NA

Missing order amounts to fill: 112


Negative delivery minutes masked: 105


In [13]:
# Step 6: Fill missing delivery minutes for completed orders by product category
completed_orders = clean_orders_copy["status"] == "completed"
completed_missing_before = clean_orders_copy.loc[
    completed_orders, "delivery_minutes"
].isna().sum()
clean_orders_copy.loc[completed_orders, "delivery_minutes"] = (
    clean_orders_copy.loc[completed_orders, "delivery_minutes"]
    .fillna(
        clean_orders_copy.loc[completed_orders]
        .groupby("category")["delivery_minutes"]
        .transform("median")
    )
)
completed_missing_after = clean_orders_copy.loc[
    completed_orders, "delivery_minutes"
].isna().sum()
print("Completed delivery times filled:", completed_missing_before - completed_missing_after)

# Step 7: Cancelled orders must have missing delivery minutes
cancelled_orders = clean_orders_copy["status"] == "cancelled"
print("Cancelled delivery values cleared:", clean_orders_copy.loc[
    cancelled_orders, "delivery_minutes"
].notna().sum())
clean_orders_copy.loc[cancelled_orders, "delivery_minutes"] = pd.NA

# Remove temporary product information
clean_orders_copy = clean_orders_copy.drop(columns=["category", "price_inr"])

Completed delivery times filled: 774
Cancelled delivery values cleared: 0


In [14]:
# Final verification
print("\nCleaned orders shape:", clean_orders_copy.shape)
print("\nOrder status counts:")
print(clean_orders_copy["status"].value_counts())
print("\nUnique order IDs:", clean_orders_copy["order_id"].nunique())
print("\nMissing order amounts:", clean_orders_copy["order_amount"].isnull().sum())
print("\nMissing delivery minutes in completed orders:", clean_orders_copy.loc[
    clean_orders_copy["status"] == "completed", "delivery_minutes"
].isnull().sum())
print("\nMissing delivery minutes in cancelled orders:", clean_orders_copy.loc[
    clean_orders_copy["status"] == "cancelled", "delivery_minutes"
].isnull().sum())
print("\nCompleted order amount total INR:", clean_orders_copy.loc[
    clean_orders_copy["status"] == "completed", "order_amount"
].sum().round(2))


Cleaned orders shape: (12309, 9)

Order status counts:
status
completed    11110
cancelled     1199
Name: count, dtype: int64

Unique order IDs: 12309

Missing order amounts: 0

Missing delivery minutes in completed orders: 0

Missing delivery minutes in cancelled orders: 1199

Completed order amount total INR: 2998668.8


In [15]:
# Save cleaned datasets
os.makedirs("data/clean", exist_ok=True)
customers_cleaned.to_csv("data/clean/customers.csv", index=False)
print("Cleaned customers dataset saved successfully")
clean_orders_copy.to_csv("data/clean/orders.csv", index=False)
print("Cleaned orders dataset saved successfully")
products.to_csv("data/clean/products.csv", index=False)
print("Cleaned products dataset saved successfully")

# Verify saved CSV shapes
clean_customers = pd.read_csv("data/clean/customers.csv")
clean_products = pd.read_csv("data/clean/products.csv")
clean_orders = pd.read_csv("data/clean/orders.csv")
print("\nCustomers shape:", clean_customers.shape)
print("Products shape:", clean_products.shape)
print("Orders shape:", clean_orders.shape)

Cleaned customers dataset saved successfully
Cleaned orders dataset saved successfully
Cleaned products dataset saved successfully

Customers shape: (1200, 5)
Products shape: (40, 4)
Orders shape: (12309, 9)


## Task 5. Store in SQLite and validate
The clean tables are written to `data/instamart.db` (tables `customers`, `products`, `orders`). The 15 checkpoints below are **assertions**: if one fails the notebook stops instead of silently passing bad data on to Part B.

In [16]:
# Import required libraries
import pandas as pd
import sqlite3
import os

# Step 1: Create required directories
os.makedirs("data/clean", exist_ok=True)
os.makedirs("data", exist_ok=True)

# Step 2: Load cleaned datasets
customers = pd.read_csv("data/clean/customers.csv")
products = pd.read_csv("data/clean/products.csv")
orders = pd.read_csv("data/clean/orders.csv")

# Step 3: Ensure order_ts is in required format
orders["order_ts"] = pd.to_datetime(
    orders["order_ts"]
).dt.strftime("%Y-%m-%d %H:%M:%S")

# Step 4: Save clean datasets as CSV
customers.to_csv("data/clean/customers.csv", index=False)
products.to_csv("data/clean/products.csv", index=False)
orders.to_csv("data/clean/orders.csv", index=False)

# Step 5: Create SQLite database
connection = sqlite3.connect("data/instamart.db")

# Step 6: Store datasets as SQLite tables
customers.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

products.to_sql(
    "products",
    connection,
    if_exists="replace",
    index=False
)

orders.to_sql(
    "orders",
    connection,
    if_exists="replace",
    index=False
)

12309

In [17]:
# ==========================================
# VALIDATION CHECKPOINTS
# ==========================================

# Check 1: Customers row count
assert pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM customers",
    connection
)["count"][0] == 1200

# Check 2: Products row count
assert pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM products",
    connection
)["count"][0] == 40

# Check 3: Orders row count
assert pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM orders",
    connection
)["count"][0] == 12309

# Check 4: Unique order IDs
assert pd.read_sql_query(
    "SELECT COUNT(DISTINCT order_id) AS count FROM orders",
    connection
)["count"][0] == 12309

# Check 5: Completed orders
assert pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM orders WHERE status='completed'",
    connection
)["count"][0] == 11110

# Check 6: Cancelled orders
assert pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM orders WHERE status='cancelled'",
    connection
)["count"][0] == 1199

# Check 7: No pending orders
assert pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM orders WHERE status='pending'",
    connection
)["count"][0] == 0

# Check 8: No missing order amounts
assert pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM orders WHERE order_amount IS NULL",
    connection
)["count"][0] == 0

# Check 9: No missing delivery time in completed orders
assert pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM orders
    WHERE status='completed' AND delivery_minutes IS NULL
    """,
    connection
)["count"][0] == 0

# Check 10: Cancelled orders have missing delivery time
assert pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM orders
    WHERE status='cancelled' AND delivery_minutes IS NULL
    """,
    connection
)["count"][0] == 1199

# Check 11: Completed revenue
revenue = pd.read_sql_query(
    """
    SELECT SUM(order_amount) AS total
    FROM orders
    WHERE status='completed'
    """,
    connection
)["total"][0]

assert round(revenue, 2) == 2998668.80

# Check 12: No missing customer ages
assert pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM customers WHERE age IS NULL",
    connection
)["count"][0] == 0

# Check 13: No missing customer genders
assert pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM customers WHERE gender IS NULL",
    connection
)["count"][0] == 0

# Check 14: Customer age minimum
assert pd.read_sql_query(
    "SELECT MIN(age) AS minimum FROM customers",
    connection
)["minimum"][0] == 18

# Check 15: Customer age maximum
assert pd.read_sql_query(
    "SELECT MAX(age) AS maximum FROM customers",
    connection
)["maximum"][0] == 60

In [18]:
# Step 7: Close database connection
connection.close()

# Print exact success message
print("ALL CHECKPOINTS PASSED")

ALL CHECKPOINTS PASSED


## Task 6. SQL analysis
The seven queries live in `sql/queries.sql` (WHERE/GROUP BY/HAVING, three-table JOIN, scalar subquery, LEFT JOIN, CTE, window functions for ranking, window function for cumulative revenue). The cell below reads the file and runs every query against `data/instamart.db`.

In [19]:
import sqlite3
import pandas as pd

# Connect to the Instamart database
connection = sqlite3.connect("data/instamart.db")

# Read the SQL file
with open("sql/queries.sql", "r", encoding="utf-8") as file:
    sql_script = file.read()

# Separate and execute each SQL query
# Remove SQL comment lines first so a ";" inside a comment cannot split a query
sql_script = "\n".join(
    line for line in sql_script.splitlines() if not line.strip().startswith("--")
)
queries = sql_script.split(";")

for i, query in enumerate(queries, start=1):
    if query.strip():
        print(f"\n========== QUERY {i} ==========")
        result = pd.read_sql_query(query, connection)
        print(result)

# Close database connection
connection.close()


========== QUERY 1 ==========
   customer_id  total_spend
0        C0693     13776.55
1        C0082     12317.70
2        C0743     12266.85
3        C0525     12241.30
4        C0291     12150.70
5        C0267     12150.25
6        C0488     11825.50
7        C0699     11693.90
8        C0512     11617.65
9        C0332     10779.50
10       C0543     10636.10
11       C0775     10263.55
12       C1024     10160.10
13       C0426      9884.60
14       C0308      9872.40
15       C1147      9740.80
16       C0407      9451.75
17       C0930      9194.00
18       C0130      9170.40
19       C0147      9141.50
20       C1146      8993.05
21       C1136      8983.75
22       C0696      8941.75
23       C0167      8718.55
24       C0819      8695.30
25       C0942      8643.60
26       C1000      8462.20
27       C0916      8395.35
28       C0454      8299.15
29       C0424      8294.20

========== QUERY 2 ==========
         city       category    revenue
0   Bengaluru    Electronics  

## Task 7. Exploratory analysis and SQL ↔ pandas cross-check
City × category revenue pivot, city-wise revenue, distribution of completed order amounts (mean vs median, skewness), IQR outliers and a correlation matrix. The pandas grand total must equal the SQL total revenue.

In [20]:
import pandas as pd
import sqlite3

# Load cleaned data
orders = pd.read_csv("data/clean/orders.csv")
customers = pd.read_csv("data/clean/customers.csv")

# Convert order timestamp
orders["order_ts"] = pd.to_datetime(orders["order_ts"])

# Filter completed orders
completed_orders = orders[
    orders["status"] == "completed"
].copy()

In [21]:
# ============================================================
# 1. PANDAS CROSS-CHECK: REVENUE BY CITY AND CATEGORY
# ============================================================

orders_customers = completed_orders.merge(
    customers[["customer_id", "city"]],
    on="customer_id"
)

orders_customers_products = orders_customers.merge(
    pd.read_csv("data/clean/products.csv")[["product_id", "category"]],
    on="product_id"
)

revenue_pivot = pd.pivot_table(
    orders_customers_products,
    values="order_amount",
    index="city",
    columns="category",
    aggfunc="sum",
    margins=True,
    fill_value=0
)

print("\n========== REVENUE PIVOT ==========")
print(revenue_pivot)

# ============================================================
# 2. CITY-WISE REVENUE ANALYSIS
# ============================================================

city_analysis = orders_customers.groupby("city").agg(
    total_revenue=("order_amount", "sum"),
    average_order_value=("order_amount", "mean"),
    completed_orders=("order_id", "count")
)

print("\n========== CITY ANALYSIS ==========")
print(city_analysis)


========== REVENUE PIVOT ==========
category       Dairy  Electronics    Grocery  Personal Care     Snacks  \
city                                                                     
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   
All        531938.95   1219248.10  614293.30      168433.95  464754.50   

category          All  
city                   
Bengaluru   523643.40  
Chennai     463629.15  
Delhi       502829.80  
Hyderabad   528589.90  
Mumbai      479058.50  
Pune        500918.05  
All        2998668.80  

========== CITY ANALYSIS ==========
           total_revenue  average_order_val

In [22]:
# ============================================================
# 3. COMPLETED ORDER AMOUNT STATISTICS
# ============================================================

print("\n========== ORDER AMOUNT STATISTICS ==========")
print(completed_orders["order_amount"].describe())

print("\nMean:", completed_orders["order_amount"].mean())
print("Median:", completed_orders["order_amount"].median())
print("Skewness:", completed_orders["order_amount"].skew())

# ============================================================
# 4. OUTLIER DETECTION USING IQR
# ============================================================

Q1 = completed_orders["order_amount"].quantile(0.25)
Q3 = completed_orders["order_amount"].quantile(0.75)

IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = completed_orders[
    (completed_orders["order_amount"] < lower_fence) |
    (completed_orders["order_amount"] > upper_fence)
]

print("\n========== OUTLIER ANALYSIS ==========")
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Fence:", lower_fence)
print("Upper Fence:", upper_fence)
print("Outlier Count:", len(outliers))


========== ORDER AMOUNT STATISTICS ==========
count    11110.000000
mean       269.907183
std        432.069126
min         12.000000
25%         64.800000
50%        135.000000
75%        297.000000
max       3998.000000
Name: order_amount, dtype: float64

Mean: 269.9071827182718
Median: 135.0
Skewness: 4.608827438881415

========== OUTLIER ANALYSIS ==========
Q1: 64.8
Q3: 297.0
IQR: 232.2
Lower Fence: -283.49999999999994
Upper Fence: 645.3
Outlier Count: 897


In [23]:
# ============================================================
# 5. CORRELATION ANALYSIS
# ============================================================

correlation = completed_orders[
    ["order_amount", "discount_pct", "quantity", "delivery_minutes"]
].corr()

print("\n========== CORRELATION MATRIX ==========")
print(correlation)
# Explicitly count outliers beyond both fences

lower_outliers = completed_orders[
    completed_orders["order_amount"] < lower_fence
]

upper_outliers = completed_orders[
    completed_orders["order_amount"] > upper_fence
]

print("Lower outliers:", len(lower_outliers))
print("Upper outliers:", len(upper_outliers))


========== CORRELATION MATRIX ==========
                  order_amount  discount_pct  quantity  delivery_minutes
order_amount          1.000000     -0.095106  0.087294         -0.003135
discount_pct         -0.095106      1.000000 -0.000359         -0.009808
quantity              0.087294     -0.000359  1.000000         -0.012623
delivery_minutes     -0.003135     -0.009808 -0.012623          1.000000
Lower outliers: 0
Upper outliers: 897


In [24]:
# Explicitly verify pivot grand total against SQL revenue

print("Pandas grand total:", revenue_pivot.loc["All", "All"])


connection = sqlite3.connect("data/instamart.db")

sql_revenue = pd.read_sql_query(
    """
    SELECT SUM(order_amount) AS total_revenue
    FROM orders
    WHERE status = 'completed'
    """,
    connection
)["total_revenue"].iloc[0]

connection.close()


print("SQL revenue:", sql_revenue)

assert round(revenue_pivot.loc["All", "All"], 2) == round(sql_revenue, 2)

# Identify the strongest correlation pair

corr = correlation.abs().copy()

# Remove diagonal values without modifying read-only arrays
corr = corr.where(~pd.DataFrame(
    corr.index.to_numpy()[:, None] == corr.columns.to_numpy(),
    index=corr.index,
    columns=corr.columns
))

strongest_pair = corr.unstack().idxmax()

print("Strongest correlation pair:", strongest_pair)

print("Correlation value:", correlation.loc[
    strongest_pair[0], strongest_pair[1]
])

Pandas grand total: 2998668.8000000003
SQL revenue: 2998668.8
Strongest correlation pair: ('order_amount', 'discount_pct')
Correlation value: -0.09510585545204023


# PART B — Customer Segments and Lapse Prediction

## Task 1. RFM table
Built from **completed** orders only, with a fixed snapshot date of `2025-10-01`:
- **Recency** – days since the last completed order
- **Frequency** – number of distinct completed orders
- **Monetary** – total completed spend

In [25]:
import pandas as pd
import numpy as np

# Load cleaned data
orders = pd.read_csv("data/clean/orders.csv")
customers = pd.read_csv("data/clean/customers.csv")

# Convert order timestamp
orders["order_ts"] = pd.to_datetime(orders["order_ts"])

# Snapshot date
snapshot = pd.Timestamp("2025-10-01 00:00:00")

# Select completed orders
completed_orders = orders[
    orders["status"] == "completed"
].copy()

# Build RFM table
rfm = completed_orders.groupby("customer_id").agg(
    last_order_ts=("order_ts", "max"),
    frequency=("order_id", "nunique"),
    monetary=("order_amount", "sum")
).reset_index()

# Calculate recency
rfm["recency"] = (
    snapshot - rfm["last_order_ts"]
).dt.days

# Remove last order timestamp
rfm = rfm[
    ["customer_id", "recency", "frequency", "monetary"]
]

# Display RFM results
print("\n========== RFM TABLE ==========")
print(rfm.head())

print("\n========== RFM SUMMARY ==========")
print(rfm[["recency", "frequency", "monetary"]].describe())

print("\nCustomer count:", len(rfm))
print("Recency median:", rfm["recency"].median())
print("Recency minimum:", rfm["recency"].min())
print("Recency maximum:", rfm["recency"].max())
print("Frequency mean:", rfm["frequency"].mean())
print("Frequency maximum:", rfm["frequency"].max())
print("Monetary mean:", rfm["monetary"].mean())
print("Monetary maximum:", rfm["monetary"].max())


========== RFM TABLE ==========
  customer_id  recency  frequency  monetary
0       C0001       14          4    512.85
1       C0002       28          7   4644.90
2       C0003        7         22   3644.85
3       C0004        0          5    369.00
4       C0005      147          2    324.00

========== RFM SUMMARY ==========
           recency    frequency      monetary
count  1175.000000  1175.000000   1175.000000
mean     33.558298     9.455319   2552.058553
std      38.433705     8.009971   2367.305104
min       0.000000     1.000000     12.000000
25%       6.000000     3.000000    505.450000
50%      19.000000     5.000000   2286.500000
75%      46.000000    17.000000   3751.525000
max     180.000000    33.000000  14766.550000

Customer count: 1175
Recency median: 19.0
Recency minimum: 0
Recency maximum: 180
Frequency mean: 9.45531914893617
Frequency maximum: 33
Monetary mean: 2552.0585531914894
Monetary maximum: 14766.550000000001


## Task 2. K-means: choosing K
RFM features are standardised (K-means uses distances, so a large-scale feature such as `monetary` would otherwise dominate). K = 2 … 8 are compared using inertia (elbow) and silhouette score.

In [26]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Select RFM features
rfm_features = rfm[
    ["recency", "frequency", "monetary"]
]

# Standardise RFM features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

# Store clustering results
cluster_results = []

# Test K values from 2 to 8
for k in range(2, 9):

    kmeans = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    )

    labels = kmeans.fit_predict(rfm_scaled)

    inertia = kmeans.inertia_

    silhouette = silhouette_score(
        rfm_scaled,
        labels
    )

    cluster_results.append({
        "K": k,
        "Inertia": inertia,
        "Silhouette": silhouette
    })

# Display results
cluster_results_df = pd.DataFrame(cluster_results)

print("\n========== K-MEANS RESULTS ==========")
print(cluster_results_df.round(3))


========== K-MEANS RESULTS ==========
   K   Inertia  Silhouette
0  2  1820.761       0.465
1  3  1268.235       0.445
2  4   743.071       0.526
3  5   617.367       0.480
4  6   503.171       0.482
5  7   408.460       0.416
6  8   366.503       0.412


## Task 3. Final segments (K = 4), PCA and segment names
K = 4 is used for the final model because it gives interpretable business groups. PCA reduces the three scaled RFM features to two components to see how much variance two dimensions keep. The segment with the highest mean spend is flagged as **Premium**.

In [27]:
from sklearn.decomposition import PCA
import os

# ============================================================
# TASK 3: FINAL CLUSTERING WITH K = 4
# ============================================================

final_kmeans = KMeans(
    n_clusters=4,
    n_init=10,
    random_state=42
)

rfm["segment_id"] = final_kmeans.fit_predict(rfm_scaled)

# Profile each segment
segment_profile = rfm.groupby("segment_id").agg(
    customer_count=("customer_id", "count"),
    mean_recency=("recency", "mean"),
    mean_frequency=("frequency", "mean"),
    mean_monetary=("monetary", "mean")
)

print("\n========== SEGMENT PROFILE ==========")
print(segment_profile.round(2))


========== SEGMENT PROFILE ==========
            customer_count  mean_recency  mean_frequency  mean_monetary
segment_id                                                             
0                      436         28.86            3.77        1057.62
1                      138         24.12            6.77        7129.70
2                      419          8.77           19.40        3410.02
3                      182        109.04            2.20         685.99


In [28]:
# ============================================================
# PCA: REDUCE RFM TO TWO DIMENSIONS
# ============================================================

pca = PCA(n_components=2)

rfm_pca = pca.fit_transform(rfm_scaled)

print("\n========== PCA RESULTS ==========")
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())


========== PCA RESULTS ==========
Explained variance ratio: [0.63427875 0.21803762]
Total explained variance: 0.8523163667502168


In [29]:
# ============================================================
# NAME SEGMENTS
# ============================================================

# Identify premium segment
premium_segment = segment_profile["mean_monetary"].idxmax()

# Assign business names based on segment profiles
segment_names = {}

for segment_id, profile in segment_profile.iterrows():

    if segment_id == premium_segment:
        segment_names[segment_id] = "Premium Customers"

    elif profile["mean_recency"] > rfm["recency"].mean():
        segment_names[segment_id] = "Inactive Customers"

    elif profile["mean_frequency"] > rfm["frequency"].mean():
        segment_names[segment_id] = "Frequent Customers"

    else:
        segment_names[segment_id] = "Occasional Customers"

# Assign segment names
rfm["segment_name"] = rfm["segment_id"].map(segment_names)

# Flag premium customers
rfm["is_premium"] = (
    rfm["segment_id"] == premium_segment
)

print("\nPremium segment ID:", premium_segment)


Premium segment ID: 1


In [30]:
# ============================================================
# SAVE CUSTOMER SEGMENTS
# ============================================================

os.makedirs("outputs", exist_ok=True)

customer_segments = rfm[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary",
        "segment_id",
        "segment_name",
        "is_premium"
    ]
]

customer_segments.to_csv(
    "outputs/customer_segments.csv",
    index=False
)

print("\nCustomer segments saved successfully.")
print(customer_segments.head())


Customer segments saved successfully.
  customer_id  recency  frequency  monetary  segment_id          segment_name  \
0       C0001       14          4    512.85           0  Occasional Customers   
1       C0002       28          7   4644.90           1     Premium Customers   
2       C0003        7         22   3644.85           2    Frequent Customers   
3       C0004        0          5    369.00           0  Occasional Customers   
4       C0005      147          2    324.00           3    Inactive Customers   

   is_premium  
0       False  
1        True  
2       False  
3       False  
4       False  


## Task 4. Lapse-prediction dataset (no leakage)
- **Cutoff:** `2025-09-01`. Features use only completed orders **before** the cutoff.
- **Cohort:** customers with at least one completed order before the cutoff.
- **Label:** `lapsed = 1` if the customer has **no** completed order in September 2025.

September orders are used for the label only. If they entered the features the model would see the answer (data leakage).

In [31]:
# ==========================================
# TASK 4: LAPSE PREDICTION DATASET
# ==========================================

# Define cutoff date and lapse observation period
cutoff = pd.Timestamp("2025-09-01")
observation_end = pd.Timestamp("2025-10-01")

# Completed orders before cutoff
pre_cutoff_orders = completed_orders[
    completed_orders["order_ts"] < cutoff
].copy()

# Customers who purchased before cutoff
lapse_cohort = pre_cutoff_orders["customer_id"].unique()

# Customers who purchased during September
september_orders = completed_orders[
    (completed_orders["order_ts"] >= cutoff) &
    (completed_orders["order_ts"] < observation_end)
]

september_customers = september_orders["customer_id"].unique()

# Create cohort dataset
lapse_data = pd.DataFrame({
    "customer_id": lapse_cohort
})

# Label customers who did not purchase during September
lapse_data["lapsed"] = (
    ~lapse_data["customer_id"].isin(september_customers)
).astype(int)

# Display results
print("\n========== LAPSE COHORT RESULTS ==========")
print("Cohort size:", len(lapse_data))
print("Lapsed customers:", lapse_data["lapsed"].sum())
print("Lapse percentage:", round(lapse_data["lapsed"].mean() * 100, 2))
print("\nTarget distribution:")
print(lapse_data["lapsed"].value_counts())


========== LAPSE COHORT RESULTS ==========
Cohort size: 1156
Lapsed customers: 443
Lapse percentage: 38.32

Target distribution:
lapsed
0    713
1    443
Name: count, dtype: int64


In [32]:
# ==========================================
# TASK 4: CUSTOMER FEATURES
# ==========================================

# Recency, frequency and monetary before cutoff
customer_features = pre_cutoff_orders.groupby("customer_id").agg(
    last_order=("order_ts", "max"),
    frequency=("order_id", "nunique"),
    monetary=("order_amount", "sum"),
    avg_discount=("discount_pct", "mean"),
    avg_delivery=("delivery_minutes", "mean")
).reset_index()

# Recency calculated from cutoff date
customer_features["recency"] = (
    cutoff - customer_features["last_order"]
).dt.days

# Cancellation rate before cutoff
pre_cutoff_all = orders[orders["order_ts"] < cutoff].copy()

cancellation_rate = pre_cutoff_all.groupby("customer_id").agg(
    total_orders=("order_id", "nunique"),
    cancelled_orders=("status", lambda x: (x == "cancelled").sum())
).reset_index()

cancellation_rate["cancellation_rate"] = (
    cancellation_rate["cancelled_orders"] /
    cancellation_rate["total_orders"]
)

# Merge cancellation rate
customer_features = customer_features.merge(
    cancellation_rate[["customer_id", "cancellation_rate"]],
    on="customer_id",
    how="left"
)

# Customer details
customer_features = customer_features.merge(
    customers[["customer_id", "city", "age", "gender", "signup_date"]],
    on="customer_id",
    how="left"
)

# Customer tenure as of cutoff
customer_features["signup_date"] = pd.to_datetime(
    customer_features["signup_date"]
)

customer_features["tenure_days"] = (
    cutoff - customer_features["signup_date"]
).dt.days

# Load products
products = pd.read_csv("data/clean/products.csv")

# Add product category to pre-cutoff orders
pre_cutoff_orders = pre_cutoff_orders.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)

# Find most frequently purchased category for each customer
favorite_category = pre_cutoff_orders.groupby("customer_id")[
    "category"
].agg(lambda x: x.mode().iloc[0]).reset_index()

favorite_category.columns = ["customer_id", "favorite_category"]


# Merge customer features and favorite category
customer_features = customer_features.merge(
    favorite_category,
    on="customer_id",
    how="left"
)

# Merge target label
lapse_dataset = customer_features.merge(
    lapse_data[["customer_id", "lapsed"]],
    on="customer_id",
    how="inner"
)

print("\nLAPSE DATASET COLUMNS:")
print(lapse_dataset.columns.tolist())


LAPSE DATASET COLUMNS:
['customer_id', 'last_order', 'frequency', 'monetary', 'avg_discount', 'avg_delivery', 'recency', 'cancellation_rate', 'city', 'age', 'gender', 'signup_date', 'tenure_days', 'favorite_category', 'lapsed']


### Train / test split
Stratified 80/20 split so both sets keep the same lapse rate. The test set stays untouched until final evaluation.

In [33]:
# ==========================================
# TASK 4: TRAIN-TEST SPLIT
# ==========================================

from sklearn.model_selection import train_test_split

# Define features and target
X = lapse_dataset.drop(
    columns=["customer_id", "last_order", "signup_date", "lapsed"]
)

y = lapse_dataset["lapsed"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n========== TRAIN-TEST SPLIT ==========")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


========== TRAIN-TEST SPLIT ==========
Training rows: 924
Testing rows: 232

Training target distribution:
lapsed
0    570
1    354
Name: count, dtype: int64

Testing target distribution:
lapsed
0    143
1     89
Name: count, dtype: int64


## Task 5. Preprocessing
A `ColumnTransformer` scales the numeric columns and one-hot encodes the categorical columns. It is fitted on the **training rows only** and only *applied* to the test rows. The check at the end compares the scaler mean with the full-cohort mean: they differ slightly, which proves the test data did not influence the scaler.

In [34]:
# ==========================================
# TASK 5: PREPROCESSING
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Numerical and categorical columns
numeric_features = [
    "recency",
    "frequency",
    "monetary",
    "avg_discount",
    "cancellation_rate",
    "avg_delivery",
    "tenure_days",
    "age"
]

categorical_features = [
    "city",
    "gender",
    "favorite_category"
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"),
         categorical_features)
    ]
)

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data without fitting
X_test_processed = preprocessor.transform(X_test)

print("\n========== PREPROCESSING RESULTS ==========")
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

print("\nMissing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

# Compare scaler training mean with full-cohort mean
scaler_frequency_mean = preprocessor.named_transformers_["num"].mean_[1]

full_cohort_frequency_mean = lapse_dataset["frequency"].mean()

print("\nScaler frequency mean:", scaler_frequency_mean)
print("Full cohort frequency mean:", full_cohort_frequency_mean)


========== PREPROCESSING RESULTS ==========
Training shape: (924, 19)
Testing shape: (232, 19)

Missing values in X_train: 0
Missing values in X_test: 0

Scaler frequency mean: 8.114718614718615
Full cohort frequency mean: 8.028546712802768


## Task 6. Models: Logistic Regression and Random Forest
Both models are wrapped in a `Pipeline` with the preprocessor and tuned with 5-fold `GridSearchCV`, scored on **F1** of the lapsed class (accuracy alone is misleading because the classes are imbalanced). The tuned models are compared with a baseline that predicts "nobody lapses".

In [35]:
# ==========================================
# TASK 6: LOGISTIC REGRESSION
# ==========================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Logistic Regression pipeline
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Hyperparameter grid
lr_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

# Grid Search
lr_grid = GridSearchCV(
    lr_pipeline,
    lr_param_grid,
    cv=5,
    scoring="f1"
)

lr_grid.fit(X_train, y_train)

print("\n========== LOGISTIC REGRESSION ==========")
print("Best parameters:", lr_grid.best_params_)
print("Best CV F1:", lr_grid.best_score_)


========== LOGISTIC REGRESSION ==========
Best parameters: {'model__C': 100}
Best CV F1: 0.650552962900789


In [36]:
# ==========================================
# TASK 6: RANDOM FOREST
# ==========================================

from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf_param_grid = {
    "model__max_depth": [3, 5, 8],
    "model__min_samples_leaf": [5, 10, 20]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("\n========== RANDOM FOREST ==========")
print("Best parameters:", rf_grid.best_params_)
print("Best CV F1:", rf_grid.best_score_)


========== RANDOM FOREST ==========
Best parameters: {'model__max_depth': 8, 'model__min_samples_leaf': 5}
Best CV F1: 0.6898463817581464


In [37]:
# ==========================================
# TASK 6: MODEL EVALUATION
# ==========================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score
)

# Best models
best_lr = lr_grid.best_estimator_
best_rf = rf_grid.best_estimator_

# Predictions on untouched test data
lr_predictions = best_lr.predict(X_test)
rf_predictions = best_rf.predict(X_test)

# Baseline model: predict every customer as not lapsed (0)
baseline_predictions = np.zeros(len(y_test), dtype=int)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_f1 = f1_score(y_test, baseline_predictions, zero_division=0)

print("\n========== BASELINE MODEL ==========")
print("Baseline Accuracy:", baseline_accuracy)
print("Baseline Lapsed-Class F1:", baseline_f1)

# Logistic Regression results
print("\n========== LOGISTIC REGRESSION TEST ==========")
print("Accuracy:", accuracy_score(y_test, lr_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_predictions))
print(classification_report(y_test, lr_predictions, zero_division=0))

# Random Forest results
print("\n========== RANDOM FOREST TEST ==========")
print("Accuracy:", accuracy_score(y_test, rf_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_predictions))
print(classification_report(y_test, rf_predictions, zero_division=0))

# Compare baseline and tuned models
print("\n========== MODEL COMPARISON ==========")
print("Baseline Accuracy:", baseline_accuracy)
print("Baseline F1:", baseline_f1)
print("Logistic Regression F1:", f1_score(y_test, lr_predictions))
print("Random Forest F1:", f1_score(y_test, rf_predictions))


========== BASELINE MODEL ==========
Baseline Accuracy: 0.6163793103448276
Baseline Lapsed-Class F1: 0.0

========== LOGISTIC REGRESSION TEST ==========
Accuracy: 0.728448275862069
Confusion Matrix:
[[111  32]
 [ 31  58]]
              precision    recall  f1-score   support

           0       0.78      0.78      0.78       143
           1       0.64      0.65      0.65        89

    accuracy                           0.73       232
   macro avg       0.71      0.71      0.71       232
weighted avg       0.73      0.73      0.73       232


========== RANDOM FOREST TEST ==========
Accuracy: 0.6853448275862069
Confusion Matrix:
[[105  38]
 [ 35  54]]
              precision    recall  f1-score   support

           0       0.75      0.73      0.74       143
           1       0.59      0.61      0.60        89

    accuracy                           0.69       232
   macro avg       0.67      0.67      0.67       232
weighted avg       0.69      0.69      0.69       232


==========

## Task 7. Handling imbalance: random oversampling
Duplicate random lapsed rows **in the training set only** until both classes are the same size, retrain Logistic Regression and compare precision, recall and F1 on the original (untouched) test set.

In [38]:
# ==========================================
# TASK 7: OVERSAMPLING
# ==========================================

from sklearn.utils import resample
from sklearn.metrics import precision_score, recall_score, f1_score

# Combine training features and target
train_data = X_train.copy()
train_data["lapsed"] = y_train.values

# Separate majority and minority classes
majority = train_data[train_data["lapsed"] == 0]
minority = train_data[train_data["lapsed"] == 1]

# Calculate how many extra lapsed rows are needed
extra_samples = len(majority) - len(minority)

# Randomly duplicate only the required number of lapsed rows
minority_extra = resample(
    minority,
    replace=True,
    n_samples=extra_samples,
    random_state=42
)

# Keep all original rows and add the extra lapsed rows
balanced_train = pd.concat(
    [majority, minority, minority_extra]
)

X_train_balanced = balanced_train.drop(columns=["lapsed"])
y_train_balanced = balanced_train["lapsed"]

print("\n========== OVERSAMPLING RESULTS ==========")
print(y_train_balanced.value_counts())

# Train Logistic Regression on balanced data
oversampled_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C=lr_grid.best_params_["model__C"],
        max_iter=1000
    ))
])

oversampled_lr.fit(X_train_balanced, y_train_balanced)

# Evaluate on original test data
oversampled_predictions = oversampled_lr.predict(X_test)

print("\n========== OVERSAMPLING COMPARISON ==========")

print("\nOriginal Tuned Logistic Regression:")
print("Precision:", precision_score(y_test, lr_predictions))
print("Recall:", recall_score(y_test, lr_predictions))
print("F1:", f1_score(y_test, lr_predictions))

print("\nOversampled Logistic Regression:")
print("Precision:", precision_score(y_test, oversampled_predictions))
print("Recall:", recall_score(y_test, oversampled_predictions))
print("F1:", f1_score(y_test, oversampled_predictions))


========== OVERSAMPLING RESULTS ==========
lapsed
0    570
1    570
Name: count, dtype: int64

========== OVERSAMPLING COMPARISON ==========

Original Tuned Logistic Regression:
Precision: 0.6444444444444445
Recall: 0.651685393258427
F1: 0.6480446927374302

Oversampled Logistic Regression:
Precision: 0.5891472868217055
Recall: 0.8539325842696629
F1: 0.6972477064220184


## Task 8. Feature importance, saved model and retention list
- Top-5 Random Forest features, matched with the sign of the Logistic Regression coefficient and a plain-English reading.
- The selected model is saved with `joblib`, reloaded, and the predictions are asserted identical.
- Retention list: the 10 **Premium** customers in the test set with the highest predicted lapse probability.

> The Premium flag comes from whole-window RFM (it uses September orders), so this retention list is **illustrative** only and is not a leakage-free evaluation.

In [39]:
# ==========================================
# TASK 8: FEATURE IMPORTANCE
# ==========================================

# Random Forest feature importance
rf_model = best_rf.named_steps["model"]

feature_names = best_rf.named_steps[
    "preprocessor"
].get_feature_names_out()

rf_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n========== TOP 5 RF FEATURES ==========")
print(rf_importance.head(5))

# Logistic Regression coefficients
lr_model = best_lr.named_steps["model"]

lr_feature_names = best_lr.named_steps[
    "preprocessor"
].get_feature_names_out()

lr_coefficients = pd.DataFrame({
    "feature": lr_feature_names,
    "coefficient": lr_model.coef_[0]
})

# Select top 5 Random Forest features
top_5_features = rf_importance.head(5).copy()

# Match RF features with LR coefficients
top_5_features = top_5_features.merge(
    lr_coefficients,
    on="feature",
    how="left",
    validate="one_to_one"
)

# Identify LR coefficient direction
top_5_features["lr_sign"] = np.where(
    top_5_features["coefficient"] > 0,
    "Positive",
    np.where(
        top_5_features["coefficient"] < 0,
        "Negative",
        "Zero"
    )
)

# Feature-specific business descriptions
feature_meanings = {
    "recency": "Days since last order",
    "frequency": "Number of orders",
    "monetary": "Total customer spending",
    "avg_discount": "Average discount received",
    "cancellation_rate": "Proportion of cancelled orders",
    "avg_delivery": "Average delivery time",
    "tenure_days": "Days since signup",
    "age": "Customer age"
}

# Generate one business interpretation per feature
def explain_feature(row):
    feature = row["feature"].split("__")[-1]
    meaning = feature_meanings.get(feature, feature)

    if pd.isna(row["coefficient"]):
        return f"{meaning}: LR coefficient unavailable."

    if row["coefficient"] > 0:
        direction = "higher predicted lapse risk"
    elif row["coefficient"] < 0:
        direction = "lower predicted lapse risk"
    else:
        direction = "no directional association with lapse risk"

    return (
        f"{meaning}: a higher value is associated with "
        f"{direction}."
    )

top_5_features["business_reading"] = top_5_features.apply(
    explain_feature,
    axis=1
)

# Display final feature importance comparison
print("\n========== TOP 5 RF FEATURES WITH LR SIGNS ==========")

print(
    top_5_features[
        [
            "feature",
            "importance",
            "coefficient",
            "lr_sign",
            "business_reading"
        ]
    ].to_string(index=False)
)


========== TOP 5 RF FEATURES ==========
                  feature  importance
1          num__frequency    0.268485
2           num__monetary    0.242298
0            num__recency    0.113011
5       num__avg_delivery    0.065104
4  num__cancellation_rate    0.056646

========== TOP 5 RF FEATURES WITH LR SIGNS ==========
               feature  importance  coefficient  lr_sign                                                                              business_reading
        num__frequency    0.268485    -1.698239 Negative               Number of orders: a higher value is associated with lower predicted lapse risk.
         num__monetary    0.242298     0.284573 Positive       Total customer spending: a higher value is associated with higher predicted lapse risk.
          num__recency    0.113011     0.039585 Positive         Days since last order: a higher value is associated with higher predicted lapse risk.
     num__avg_delivery    0.065104     0.078514 Positive         Average

In [40]:
# ==========================================
# TASK 8: SAVE AND RELOAD MODEL
# ==========================================

import joblib
import os

os.makedirs("outputs", exist_ok=True)

# Select the final model: the tuned model with the higher lapsed-class F1 on the test set
lr_test_f1 = f1_score(y_test, lr_predictions)
rf_test_f1 = f1_score(y_test, rf_predictions)
selected_model = best_lr if lr_test_f1 >= rf_test_f1 else best_rf
selected_name = "Tuned Logistic Regression" if selected_model is best_lr else "Tuned Random Forest"

# Generate predictions before saving
selected_predictions = selected_model.predict(X_test)

# Save the selected model
joblib.dump(selected_model, "outputs/lapse_model.joblib")

# Reload the saved model
loaded_model = joblib.load("outputs/lapse_model.joblib")

# Generate predictions after reloading
reloaded_predictions = loaded_model.predict(X_test)

# Verify predictions are identical
predictions_identical = np.array_equal(
    selected_predictions,
    reloaded_predictions
)

print("\n========== MODEL RELOAD VERIFICATION ==========")
print("Selected Model:", selected_name)
print("Predictions identical:", predictions_identical)

assert predictions_identical, "Model predictions changed after reloading!"


========== MODEL RELOAD VERIFICATION ==========
Selected Model: Tuned Logistic Regression
Predictions identical: True


In [41]:
# ==========================================
# TASK 8: RETENTION LIST
# ==========================================

# Predict lapse probability for test customers
lapse_probabilities = loaded_model.predict_proba(X_test)[:, 1]

# Create test customer results
retention_list = pd.DataFrame({
    "customer_id": lapse_dataset.loc[X_test.index, "customer_id"].values,
    "lapse_probability": lapse_probabilities,
    "actually_lapsed": y_test.values
})

# Load premium segment customers
customer_segments = pd.read_csv("outputs/customer_segments.csv")

premium_customers = customer_segments[
    customer_segments["is_premium"] == True
]["customer_id"]

# Filter premium customers from test set
retention_list = retention_list[
    retention_list["customer_id"].isin(premium_customers)
].copy()

# Rank premium customers by highest lapse probability
retention_list = retention_list.sort_values(
    "lapse_probability",
    ascending=False
)

# Select only the top 10 premium customers
retention_list = retention_list.head(10).copy()

# Save the top 10 retention list
retention_list.to_csv(
    "outputs/retention_list.csv",
    index=False
)

print("\n========== TOP 10 PREMIUM RETENTION LIST ==========")
print("Customers selected:", len(retention_list))
print("Actually lapsed:", retention_list["actually_lapsed"].sum())

print("\nTop 10 Premium Customers:")
print(retention_list.to_string(index=False))

print("\nTop 10 retention customers:")


========== TOP 10 PREMIUM RETENTION LIST ==========
Customers selected: 10
Actually lapsed: 3

Top 10 Premium Customers:
customer_id  lapse_probability  actually_lapsed
      C0525           0.604691                1
      C0267           0.585432                0
      C0082           0.581029                0
      C0060           0.570267                0
      C0167           0.551091                0
      C0271           0.521994                1
      C0942           0.478027                0
      C1146           0.475997                1
      C1156           0.471602                0
      C0785           0.466910                0

Top 10 retention customers:


## Files produced by this notebook
Notebook 2 (Part C and Part D) reads `data/instamart.db` and `outputs/customer_segments.csv`.

In [42]:
for path in ["data/clean/customers.csv", "data/clean/products.csv", "data/clean/orders.csv",
             "data/instamart.db", "outputs/customer_segments.csv",
             "outputs/lapse_model.joblib", "outputs/retention_list.csv"]:
    print(f"{'OK     ' if (ROOT / path).exists() else 'MISSING'} {path}")

OK      data/clean/customers.csv
OK      data/clean/products.csv
OK      data/clean/orders.csv
OK      data/instamart.db
OK      outputs/customer_segments.csv
OK      outputs/lapse_model.joblib
OK      outputs/retention_list.csv
